# 第11章 图像金字塔（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 11 章跟随代码。**全量收录原书代码**（例 11.1~例 11.7），例题齐全，代码与网页笔记 ch11.md 一致（lena 系列素材按笔记替换：picture1_gray.jpg 顶替 lena.bmp、picture3_mouth.jpg 顶替 lenas.bmp、picture1.jpg 顶替彩色 lena）。
> `cv2.imshow` 需在**本地 Python 环境**运行（浏览器/Jupyter 内不弹窗）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg 等素材都在这里），否则 `imread` 返回 None。


## 11.1 理论基础

- **图像金字塔**（image pyramid）：同一幅图像不同分辨率的子图集合，由原图不断向下采样产生；底部第 0 层是高分辨率原始图像，每上一层宽高减半、面积变四分之一，顶层可以小到一个像素点。
- **向下采样**两种方式：直接删除偶数行和偶数列；先滤波（邻域平均→平均金字塔，高斯滤波→高斯金字塔）再删。`cv2.pyrDown` 采用高斯方式。
- **向上采样**：宽高各变为 2 倍，每列右侧、每行下方补零，再用向下采样的高斯核卷积，**核系数乘 4**（补偿四分之三像素为零导致的亮度损失）。`cv2.pyrUp` 封装了整个过程。
- 向上采样与向下采样**不可逆**：先后做两次只是尺寸复原，像素值并不一致——这正是拉普拉斯金字塔（把丢失信息存成差值图）要解决的问题。


## 11.2 pyrDown 函数及使用

`dst = cv2.pyrDown(src[, dstsize[, borderType]])`

- `src`：原始图像；`dst`：目标图像。
- `dstsize`：目标图像大小，默认 `Size((src.cols+1)/2, (src.rows+1)/2)`，须满足 |dst.width*2-src.cols|<=2、|dst.height*2-src.rows|<=2（注意 dstsize 是**（宽, 高）**格式）。
- `borderType`：边界类型，默认 BORDER_DEFAULT。
- 内部两步：先做高斯滤波（5×5 核）得到近似图像，再抛弃近似图像的偶数行和偶数列。

### 例 11.1 · 连续三次向下采样

原书素材为 lena.bmp（512×512 灰度），此处用 picture1_gray.jpg 顶替。经过向下采样后，行数和列数都变为原来的二分之一，图像整体大小变为原来的四分之一。


In [ ]:
import cv2
o=cv2.imread("picture1_gray.jpg",cv2.IMREAD_GRAYSCALE)
r1=cv2.pyrDown(o)
r2=cv2.pyrDown(r1)
r3=cv2.pyrDown(r2)
print("o.shape=",o.shape)
print("r1.shape=",r1.shape)
print("r2.shape=",r2.shape)
print("r3.shape=",r3.shape)
cv2.imshow("original",o)
cv2.imshow("r1",r1)
cv2.imshow("r2",r2)
cv2.imshow("r3",r3)
cv2.waitKey()
cv2.destroyAllWindows()


## 11.3 pyrUp 函数及使用

`dst = cv2.pyrUp(src[, dstsize[, borderType]])`

- `dstsize`：目标图像大小，默认 `Size(src.cols*2, src.rows*2)`，须满足 |dst.width-src.cols*2|<=dst.width mod 2、|dst.height-src.rows*2|<=dst.height mod 2（原书第一式的 dst.width 误写作 widh，此处已更正；满足约束的**奇数** dstsize 也合法）。
- 内部过程：补零 → 用向下采样的高斯核滤波 → 核系数乘 4。

### 例 11.2 · 连续三次向上采样

原书素材为 lenas.bmp（64×64 彩色），此处用 picture3_mouth.jpg 顶替。向上采样后宽度和高度都变为原来的 2 倍，放大出来的部分是"补零 + 滤波"平滑出来的，看上去越来越模糊。


In [ ]:
import cv2
o=cv2.imread("picture3_mouth.jpg")
r1=cv2.pyrUp(o)
r2=cv2.pyrUp(r1)
r3=cv2.pyrUp(r2)
print("o.shape=",o.shape)
print("r1.shape=",r1.shape)
print("r2.shape=",r2.shape)
print("r3.shape=",r3.shape)
cv2.imshow("original",o)
cv2.imshow("r1",r1)
cv2.imshow("r2",r2)
cv2.imshow("r3",r3)
cv2.waitKey()
cv2.destroyAllWindows()


## 11.4 采样可逆性的研究

一幅 M×N 的图像：向下采样后为 (M/2)×(N/2)，向上采样后为 (2M)×(2N)。先向下采样、再向上采样（或反过来），尺寸恢复原状，但像素值并不一致——向上采样和向下采样**不是互逆的**。

注意：`diff = up - o` 是 numpy 在 uint8 上的模 256 **回绕**运算——真实差值为负时（变暗）显示成 256-|d| 的亮值，差值图上的亮噪点不等于"差异大"；看真实差异用 `cv2.absdiff` 或把两图转成 int16/float32 再相减。

### 例 11.3 · 先向下采样，再向上采样

原书彩色 lena 用 picture1.jpg 顶替。


In [ ]:
import cv2
o=cv2.imread("picture1.jpg")
down=cv2.pyrDown(o)
up=cv2.pyrUp(down)
diff=up-o   #构造diff图像，查看up与o的区别
print("o.shape=",o.shape)
print("up.shape=",up.shape)
cv2.imshow("original",o)
cv2.imshow("up",up)
cv2.imshow("difference",diff)
cv2.waitKey()
cv2.destroyAllWindows()


### 例 11.4 · 先向上采样，再向下采样

"先上后下"的损失比"先下后上"略小（实测非零差值像素约 43.2% vs 60.8%），但依然不可逆。（原书正文把子图标号误写为"(c)图是 up…(d)图是 down 和 o 的差值"，实际图 11.10 只有 (a)(b)(c) 三个子图：(b) 是 down、(c) 是 diff。）


In [ ]:
import cv2
o=cv2.imread("picture1.jpg")
up=cv2.pyrUp(o)
down=cv2.pyrDown(up)
diff=down-o   #构造diff图像，查看down与o的区别
print("o.shape=",o.shape)
print("down.shape=",down.shape)
cv2.imshow("original",o)
cv2.imshow("down",down)
cv2.imshow("difference",diff)
cv2.waitKey()
cv2.destroyAllWindows()


## 11.5 拉普拉斯金字塔

### 11.5.1 定义

拉普拉斯金字塔第 i 层：`Li = Gi - pyrUp(Gi+1)`，其中 Gi 是高斯金字塔第 i 层。每层存的就是向下采样时丢失的信息，与 Gi 同尺寸；uint8 回绕让它看上去像"边缘纹理噪声"——这类图不是用来看的，是用来"加回去"复原的。

### 例 11.5 · 构造拉普拉斯金字塔


In [ ]:
import cv2
O=cv2.imread("picture1.jpg")
G0=O
G1=cv2.pyrDown(G0)
G2=cv2.pyrDown(G1)
G3=cv2.pyrDown(G2)
L0=G0-cv2.pyrUp(G1)
L1=G1-cv2.pyrUp(G2)
L2=G2-cv2.pyrUp(G3)
print("L0.shape=",L0.shape)
print("L1.shape=",L1.shape)
print("L2.shape=",L2.shape)
cv2.imshow("L0",L0)
cv2.imshow("L1",L1)
cv2.imshow("L2",L2)
cv2.waitKey()
cv2.destroyAllWindows()


### 11.5.2 应用

恢复关系式（由定义式 Li = Gi - pyrUp(Gi+1) 移项得到）：`Gi = Li + pyrUp(Gi+1)`——只要手里有高斯金字塔某一层 Gi+1 和对应的拉普拉斯层 Li，就能一步加回 Gi。验证误差用"先取绝对值再求和"的技巧：直接求和时 +3 与 -3 也会加出 0，看似无差异实则差异很大。

### 例 11.6 · 恢复原始图像


In [ ]:
import cv2
import numpy as np
O=cv2.imread("picture1.jpg")
G0=O
G1=cv2.pyrDown(G0)
L0=O-cv2.pyrUp(G1)
RO=L0+cv2.pyrUp(G1)  # 通过拉普拉斯图像复原的原始图像
print("O.shape=",O.shape)
print("RO.shape=",RO.shape)
result=RO-O  # 将O和RO做减法运算
# 计算result的绝对值，避免求和时正负抵消，3+(-3)=0
result=abs(result)
# 计算result所有元素的和
print("原始图像O 与恢复图像RO 之差的绝对值和：",np.sum(result))


### 例 11.7 · 恢复高斯金字塔内的多层图像

逐层复原 G0、G1、G2，误差绝对值和全为 0——零误差依赖 numpy uint8 加减的模 256 回绕算术（(a-b)+b ≡ a (mod 256) 恒成立）；把减法换成饱和的 `cv2.subtract` 立即出现百万级误差。


In [ ]:
import cv2
import numpy as np
O=cv2.imread("picture1.jpg")
#==============生成高斯金字塔===================
G0=O
G1=cv2.pyrDown(G0)
G2=cv2.pyrDown(G1)
G3=cv2.pyrDown(G2)
#============生成拉普拉斯金字塔===================
L0=G0-cv2.pyrUp(G1) #拉普拉斯金字塔第0层
L1=G1-cv2.pyrUp(G2) #拉普拉斯金字塔第1层
L2=G2-cv2.pyrUp(G3) #拉普拉斯金字塔第2层
#=================复原G0======================
RG0=L0+cv2.pyrUp(G1)  #通过拉普拉斯图像复原的原始图像G0
print("G0.shape=",G0.shape)
print("RG0.shape=",RG0.shape)
result=RG0-G0  #将RG0和G0相减
#计算result的绝对值，避免求和时正负抵消，3+(-3)=0
result=abs(result)
#计算result所有元素的和
print("原始图像G0 与恢复图像RG0 差值的绝对值和：",np.sum(result))
#=================复原G1======================
RG1=L1+cv2.pyrUp(G2) #通过拉普拉斯图像复原G1
print("G1.shape=",G1.shape)
print("RG1.shape=",RG1.shape)
result=RG1-G1  #将RG1和G1相减
print("原始图像G1 与恢复图像RG1 之差的绝对值和：",np.sum(abs(result)))
#=================复原G2======================
RG2=L2+cv2.pyrUp(G3) #通过拉普拉斯图像复原G2
print("G2.shape=",G2.shape)
print("RG2.shape=",RG2.shape)
result=RG2-G2  #将RG2和G2相减
print("原始图像G2 与恢复图像RG2 之差的绝对值和：",np.sum(abs(result)))
